In [ ]:
import os
import random
import torch
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms
from transformers import ViTForImageClassification  # only the model
from torch.optim import AdamW  # optimizer from PyTorch
from sklearn.metrics import confusion_matrix, classification_report
import numpy as np

# Device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


Using device: cuda


In [3]:
# Hyperparameters
batch_size = 16
learning_rate = 3e-5
epochs = 3

# Kaggle data directory
data_dir = "/kaggle/input/food41/images"


In [4]:
# Training transforms
train_transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225])
])

# Validation transforms
val_transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225])
])


In [6]:
# Load full dataset
full_dataset = datasets.ImageFolder(data_dir)
num_classes = len(full_dataset.classes)
print("Number of classes:", num_classes)  # should be 41

# Shuffle indices
indices = list(range(len(full_dataset)))
random.shuffle(indices)

# 80% train, 20% validation
split = int(0.8 * len(full_dataset))
train_indices = indices[:split]
val_indices = indices[split:]

# Subsets
train_dataset = Subset(full_dataset, train_indices)
val_dataset = Subset(full_dataset, val_indices)

# Apply transforms
train_dataset.dataset.transform = train_transform
val_dataset.dataset.transform = val_transform

# DataLoaders
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

# Sanity check
print("Max train label:", max([label for _, label in train_dataset]))
print("Max val label:", max([label for _, label in val_dataset]))


Number of classes: 101
Max train label: 100
Max val label: 100


In [8]:
model = ViTForImageClassification.from_pretrained(
    "google/vit-base-patch16-224",
    num_labels=num_classes,       # 101 for your dataset
    ignore_mismatched_sizes=True  # <-- crucial
)
model.to(device)


Some weights of ViTForImageClassification were not initialized from the model checkpoint at google/vit-base-patch16-224 and are newly initialized because the shapes did not match:
- classifier.bias: found shape torch.Size([1000]) in the checkpoint and torch.Size([101]) in the model instantiated
- classifier.weight: found shape torch.Size([1000, 768]) in the checkpoint and torch.Size([101, 768]) in the model instantiated
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


ViTForImageClassification(
  (vit): ViTModel(
    (embeddings): ViTEmbeddings(
      (patch_embeddings): ViTPatchEmbeddings(
        (projection): Conv2d(3, 768, kernel_size=(16, 16), stride=(16, 16))
      )
      (dropout): Dropout(p=0.0, inplace=False)
    )
    (encoder): ViTEncoder(
      (layer): ModuleList(
        (0-11): 12 x ViTLayer(
          (attention): ViTAttention(
            (attention): ViTSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
            )
            (output): ViTSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.0, inplace=False)
            )
          )
          (intermediate): ViTIntermediate(
            (dense): Linear(in_features=768, out_features=3072, bias=True)
            (intermed

In [9]:
optimizer = AdamW(model.parameters(), lr=learning_rate)
criterion = torch.nn.CrossEntropyLoss()


In [12]:
from tqdm import tqdm  # progress bar

for epoch in range(epochs):
    model.train()
    running_loss = 0

    # Wrap the train_loader with tqdm
    loop = tqdm(train_loader, leave=True, desc=f"Epoch {epoch+1}/{epochs}")

    for images, labels in loop:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images).logits
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()

        # Update tqdm description with current batch loss
        loop.set_postfix(loss=running_loss/ (loop.n+1))  # average loss so far

    print(f"Epoch {epoch+1}/{epochs}, Avg Loss: {running_loss/len(train_loader):.4f}")


Epoch 1/3: 100%|██████████| 5050/5050 [1:00:54<00:00,  1.38it/s, loss=0.128]


Epoch 1/3, Avg Loss: 0.1277


Epoch 2/3: 100%|██████████| 5050/5050 [1:00:47<00:00,  1.38it/s, loss=0.0695]


Epoch 2/3, Avg Loss: 0.0695


Epoch 3/3: 100%|██████████| 5050/5050 [1:00:53<00:00,  1.38it/s, loss=0.0513]

Epoch 3/3, Avg Loss: 0.0513


In [20]:
from tqdm import tqdm
import torch
import numpy as np
from sklearn.metrics import confusion_matrix, classification_report
import os
import random

# Set model to evaluation mode
model.eval()

all_preds = []
all_labels = []

# Use smaller batch size for evaluation if memory is tight
val_loader_eval = DataLoader(val_dataset, batch_size=16, shuffle=False)

# Wrap val_loader with tqdm to show progress and ETA
loop = tqdm(val_loader_eval, desc="Evaluating", leave=True, dynamic_ncols=True)

with torch.no_grad():
    for images, labels in loop:
        images, labels = images.to(device), labels.to(device)
        
        outputs = model(images).logits
        preds = torch.argmax(outputs, dim=1)
        
        # Move predictions and labels to CPU immediately
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
        
        # Optional: update progress bar with number of images processed
        loop.set_postfix(processed=len(all_preds))

# Convert to numpy arrays
all_preds = np.array(all_preds)
all_labels = np.array(all_labels)

# Overall accuracy
accuracy = np.mean(all_preds == all_labels)
print(f"\nValidation Accuracy: {accuracy*100:.2f}%")

# Confusion Matrix
cm = confusion_matrix(all_labels, all_preds)
print("Confusion Matrix:\n", cm)

# Classification Report
report = classification_report(all_labels, all_preds, target_names=full_dataset.classes)
print("Classification Report:\n", report)

# Example predictions
print("\nExample predictions:")
for i in range(5):
    idx = random.randint(0, len(all_labels)-1)
    print(f"Predicted: {full_dataset.classes[all_preds[idx]]}, Actual: {full_dataset.classes[all_labels[idx]]}")

# Save model checkpoint
checkpoint_dir = "/kaggle/working/checkpoints"
os.makedirs(checkpoint_dir, exist_ok=True)
torch.save(model.state_dict(), f"{checkpoint_dir}/vit_food41.pth")
print("Model saved successfully!")

# Save evaluation report to a file
with open("/kaggle/working/evaluation_report.txt", "w") as f:
    f.write(f"Validation Accuracy: {accuracy*100:.2f}%\n\n")
    f.write("Classification Report:\n")
    f.write(report)
print("Evaluation report saved at /kaggle/working/evaluation_report.txt")


Evaluating: 100%|██████████| 1263/1263 [05:56<00:00,  3.54it/s, processed=20200]



Validation Accuracy: 83.57%
Confusion Matrix:
 [[146   1   6 ...   0   1   1]
 [  0 151   0 ...   0   0   0]
 [  5   1 172 ...   0   0   0]
 ...
 [  3   0   1 ... 156   1   0]
 [  0   0   0 ...   0 150   0]
 [  2   0   0 ...   0   0 159]]
Classification Report:
                          precision    recall  f1-score   support

              apple_pie       0.58      0.70      0.64       209
         baby_back_ribs       0.77      0.86      0.81       175
                baklava       0.86      0.87      0.86       197
         beef_carpaccio       0.83      0.85      0.84       180
           beef_tartare       0.75      0.86      0.80       207
             beet_salad       0.83      0.78      0.80       202
               beignets       0.74      0.95      0.83       199
               bibimbap       0.94      0.94      0.94       200
          bread_pudding       0.78      0.59      0.67       209
      breakfast_burrito       0.76      0.85      0.80       229
             brusche

In [23]:
import os
checkpoint_dir = "/kaggle/working/checkpoints"
os.makedirs(checkpoint_dir, exist_ok=True)
torch.save(model.state_dict(), f"{checkpoint_dir}/vit_food41.pth")
print("Model saved successfully!")


Model saved successfully!
